# SAM2 GPU ベンチ（Colab 用・使い捨て）

1. メニューの **ランタイム → ランタイムのタイプを変更 → T4 GPU** を選んで保存
2. 上のセルから順に ▶ で実行
3. 最後に出る `report.txt` の中身をチャットに貼る

In [ ]:
# GPU が割り当てられているか確認（表が出れば OK）
!nvidia-smi

In [ ]:
# GPU 版 ONNX Runtime を入れる。ドライバの CUDA の世代に合わせて版を選ぶ
import re, subprocess
smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout
cuda = float(re.search(r'CUDA Version:\s*([\d.]+)', smi).group(1))
ver = '1.30.0' if cuda >= 13 else '1.25.1'
print(f'ドライバの CUDA {cuda} → onnxruntime-gpu {ver}')
!pip -q uninstall -y onnxruntime onnxruntime-gpu
!pip -q install "onnxruntime-gpu[cuda,cudnn]=={ver}" onnx==1.19.1

In [ ]:
# bench.py / convert.py / 動画(input.mp4) の3つを選んでアップロード
from google.colab import files
uploaded = files.upload()
print(list(uploaded))

In [ ]:
# 計測（10秒分・fp32 と fp16 を比較）
!python bench.py --video input.mp4 --point 0.51,0.44 --device cuda --compare --secs 10 --models models --out out

In [ ]:
# 結果を表示し、動画ごとまとめてダウンロード
print(open('out/report.txt', encoding='utf-8').read())
!cd out && zip -q -r ../result.zip .
files.download('result.zip')